# Secure distributed Computation Final Project
## Luke Holmes and Jake Pappas

We are going to implement linear regression inference on secret inputs to provide disease progression inferences to clients securely. We will use a 1-server, n-clients architecture, using OT to generate multiplication triples. Our dataset is the scikit-learn diabetes dataset.

## Setup

In [130]:
# Useful imports and utility functions
import pychor
import galois
import pandas as pd
import sklearn
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

server = pychor.Party('server')
user = pychor.Party('user')

import numpy as np

GF = galois.GF(2**61-1)

p = 2**61 - 1
SCALE = 10**6  # fixed-point scale

def encode(v: float) -> int:
    """Float -> GF integer at scale SCALE. Negatives wrap via mod p."""
    return int(round(v * SCALE)) % p

def encode_squared(v: float) -> int:
    """Float -> GF integer at scale SCALE^2. Used for the bias term."""
    return int(round(v * SCALE * SCALE)) % p

def decode_product(v) -> float:
    """GF element at scale SCALE^2 -> float. Handles sign wrap."""
    x = int(v) % p
    if x > p // 2:
        x -= p
    return x / (SCALE * SCALE)

@pychor.local_function
def share(x):
    s1 = GF.Random()
    s2 = GF(x) - s1
    return s1, s2

## Training and plaintext inference

In [129]:
# Load as a pandas dataframe
diabetes_df = server.constant(load_diabetes(as_frame=True, scaled=False))
df = server.constant(diabetes_df.val.frame)
model = LinearRegression()
df_test = df.val.iloc[:2]
@pychor.local_function
def train_model(frame):
    X = frame.drop(columns='target')
    y = frame['target']
    model = LinearRegression()
    model.fit(X, y)
    return model

@pychor.local_function
def get_coef(m, i):
    return GF(encode(m.coef_[i]))

@pychor.local_function
def get_intercept(m):
    return GF(encode_squared(m.intercept_))

with pychor.LocalBackend():
    model_located = train_model(df)

print(f"Test data:\n{df_test.drop(columns='target')}")
print(f"Coefficients: {model_located.val.coef_}")   # .val only for printing
print(f"Intercept:    {model_located.val.intercept_:.4f}")

Test data:
    age  sex   bmi     bp     s1     s2    s3   s4      s5    s6
0  59.0  2.0  32.1  101.0  157.0   93.2  38.0  4.0  4.8598  87.0
1  48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.8918  69.0
Coefficients: [-3.63612242e-02 -2.28596481e+01  5.60296209e+00  1.11680799e+00
 -1.08999633e+00  7.46450456e-01  3.72004715e-01  6.53383194e+00
  6.84831250e+01  2.80116989e-01]
Intercept:    -334.5671


In [143]:
def get_secure_features(data):
    # User secret-shares their normalized feature vector (encode handles negatives via mod p)
    x_row = data.iloc[0].values  # shape (10,)
    return [SecInt.input(user.constant(GF(encode(x_row[i])))) for i in range(10)]

def get_secure_weights():
    # Server secret-shares its model weights (encode handles negative weights correctly)
    return [SecInt.input(get_coef(model_located, server.constant(i))) for i in range(10)]

## Inference protocol

In [146]:
# @pychor.local_function
# def encode_feature(x_row, i):
#     return GF(encode(float(x_row[i])))

def infer(sec_features, sec_weights):
    gen_triples(10)

    # Dot product
    total = sec_features[0] * sec_weights[0]
    for i in range(1, 10):
        total += sec_features[i] * sec_weights[i]

    # Intercept stays at the server via get_intercept
    intercept = SecInt.input(get_intercept(model_located))
    total += intercept

    raw = total.reveal()
    return decode_product(raw.val)


with pychor.LocalBackend():
    test_features = df_test.drop(columns='target')
    secure_features = get_secure_features(test_features)
    secure_weights = get_secure_weights()
    mpc_result = infer(secure_features, secure_weights)
    plain_result = model_located.val.predict([test_features.iloc[0].values])[0]
    print(f"MPC prediction:       {mpc_result:.4f}")
    print(f"Plaintext prediction: {plain_result:.4f}")
    print(f"Error:                {abs(mpc_result - plain_result):.6f}")

MPC prediction:       206.1167
Plaintext prediction: 206.1167
Error:                0.000033


C:\Users\Jake Pappas\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## Inference on Raw input

In [148]:
# Inference on user input raw data
def get_user_data():
    """
    Prompts the user for raw indicator inputs
    Example input from dataset:
     age  sex   bmi     bp     s1     s2    s3   s4      s5    s6
    48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.8918  69.0
    """
    # Define the keys we need to collect
    keys = ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
    vitals_dict = {}

    print("Please enter the following information:")

    for key in keys:
        # Prompt user and store the result in the dictionary
        value = input(f"{key.upper()}: ")
        vitals_dict[key] = value

    return vitals_dict

user_gen_info = get_user_data()
# Wrap in a list so pandas treats the dict as a single row
user_df = pd.DataFrame([{k: float(v) for k, v in user_gen_info.items()}])

with pychor.LocalBackend():
    sec_weights = get_secure_weights()
    sec_features = get_secure_features(user_df)
    mpc_result = infer(sec_features, sec_weights)
    print(f"Predicted disease progression: {mpc_result:.2f}")


Please enter the following information:


AGE:  1
SEX:  1
BMI:  1
BP:  1
S1:  1
S2:  1
S3:  1
S4:  1
S5:  1
S6:  1


Predicted disease progression: -275.42


In [126]:
from dataclasses import dataclass

multiplication_triples = []

@dataclass
class SecInt:
    # s1 is server's share of the value, and s2 is user's share
    s1: galois.GF
    s2: galois.GF

    @classmethod
    def input(cls, val):
        """Secret share an input: server holds s1, and user holds s2"""
        s1, s2 = share(val).untup(2)
        if server in val.parties:
            s2.send(server, user)
            return SecInt(s1, s2)
        else:
            s1.send(user, server)
            return SecInt(s1, s2)

    def __add__(x, y):
        """Add two SecInt objects using local addition of shares"""
        return SecInt(x.s1 + y.s1, x.s2 + y.s2)

    def __sub__(x, y):
        """Subtract two SecInt objects using local addition of shares"""
        return SecInt(x.s1 - y.s1, x.s2 - y.s2)

    def __mul__(x, y):
        """Multiply two SecInt objects using a triple"""
        triple = multiplication_triples.pop()
        r1, r2 = protocol_mult((x.s1, x.s2), (y.s1, y.s2), triple)
        return SecInt(r1, r2)

    def reveal(self):
        """Reveal the secret value by broadcast and reconstruction"""
        self.s1.send(server, user)
        self.s2.send(user, server)
        return self.s1 + self.s2

def functionality_gen_triple():
    Fgen = pychor.Party('Fgen')

    def deal_shares(x):
        s1, s2 = share(x).untup(2)
        s1.send(Fgen, server)
        s2.send(Fgen, user)
        return s1, s2

    # Step 1: generate a, b, c
    a = Fgen.constant(GF.Random())
    b = Fgen.constant(GF.Random())
    c = a * b

    # Step 2: secret share a, b, c
    a1, a2 = deal_shares(a)
    b1, b2 = deal_shares(b)
    c1, c2 = deal_shares(c)
    return (a1, a2), (b1, b2), (c1, c2)

def protocol_mult(x, y, triple):
    x1, x2 = x
    y1, y2 = y
    (a1, a2), (b1, b2), (c1, c2) = triple

    # Step 1. server computes d_1 = x_1 - a_1 and sends the result to user
    d1 = x1 - a1
    d1.send(server, user)

    # Step 2. user computes d_2 = x_2 - a_2 and sends the result to server
    d2 = x2 - a2
    d2.send(user, server)

    # Step 3: server and user both compute $d = d_1 + d_2 = x - a$
    d = d1 + d2

    # Step 4. server computes e_1 = y_1 - b_1 and sends the result to user
    e1 = y1 - b1
    e1.send(server, user)

    # Step 5. user computes e_2 = y_2 - b_2 and sends the result to server
    e2 = y2 - b2
    e2.send(user, server)

    # Step 6. server and user both compute $e = e_1 + e_2 = y - b$
    e = e1 + e2

    # Step 7. server computes r_1 = d*e + d*b_1 + e*a_1 + c_1
    r_1 = d * e + d * b1 + e * a1 + c1

    # Step 8. user computes r_2 = 0 + d*b_2 + e*a_2 + c_2
    r_2 = d * b2 + e * a2 + c2

    return r_1, r_2

def gen_triples(n):
    for _ in range(n):
        multiplication_triples.append(functionality_gen_triple())